# EyeAI AMD3 — FastAPI AI Engine V2 Explainability Smoke Test

This notebook does not train the model. It loads the frozen Run 09 + horizontal-flip TTA model package, creates FastAPI V2 in-process, and validates `/predict-with-explanation` with persisted original, processed, heatmap, overlay, and metadata artifacts.


## 1. Repository, model-package, and artifact settings

Attach the exported `run09_tta_v1` model package and the prepared HYAMD dataset. A Kaggle GPU is strongly recommended because explanation requires one gradient-enabled transformer pass.


In [ ]:
from pathlib import Path
import importlib
import io
import json
import os
import shutil
import subprocess
import sys

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")

MODEL_PACKAGE_OVERRIDE = None
DATASET_MOUNT = Path("/kaggle/input/datasets/alihasan15/hymd-armd-dataset")
API_CONFIG = REPO_DIR / "configs/api/fastapi_v2.yaml"
EXPLANATION_OUTPUT_DIR = Path("/kaggle/working/eyeai_fastapi_v2_artifacts")

print("Prepared dataset mount:", DATASET_MOUNT)
print("Explanation output:", EXPLANATION_OUTPUT_DIR)


## 2. Clone and install the current project

The cell installs the API and inserts `src` into the active kernel path to avoid stale editable-install imports.


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

repo_src = str(REPO_DIR / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
print("Repository and package are ready.")


## 3. Resolve the frozen Run 09 model package

Only the portable inference package is needed. The original training checkpoint and RETFound base checkpoint are not required.


In [ ]:
def resolve_model_package(override=None):
    if override:
        path = Path(override)
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path

    candidates = sorted({
        version_path.parent
        for root in [Path("/kaggle/working"), Path("/kaggle/input")]
        if root.exists()
        for version_path in root.rglob("version.json")
        if (version_path.parent / "model.pth").is_file()
        and (version_path.parent / "model_config.yaml").is_file()
    })
    preferred = [path for path in candidates if path.name == "run09_tta_v1"]
    candidates = preferred or candidates
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError("No exported EyeAI model package was found.")
    raise RuntimeError(
        "Multiple model packages were found. Set MODEL_PACKAGE_OVERRIDE explicitly:
"
        + "
".join(f"- {path}" for path in candidates)
    )

MODEL_PACKAGE_DIR = resolve_model_package(MODEL_PACKAGE_OVERRIDE)
print("Model package:", MODEL_PACKAGE_DIR)
print(json.dumps(json.loads((MODEL_PACKAGE_DIR / "version.json").read_text()), indent=2))


## 4. Create FastAPI V2 and verify capability metadata

The model is loaded once. The artifact output path is overridden for the current Kaggle session.


In [ ]:
import torch
from fastapi.testclient import TestClient

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running explainability.")

os.environ["EYEAI_EXPLANATION_OUTPUT_DIR"] = str(EXPLANATION_OUTPUT_DIR)

from eyeai.api.config import ApiSettings
from eyeai.api.main import create_app

settings = ApiSettings.from_yaml(
    API_CONFIG,
    model_package_override=MODEL_PACKAGE_DIR,
    device_override="cuda",
)
app = create_app(settings)
client_context = TestClient(app)
client = client_context.__enter__()

health = client.get("/health")
model_info = client.get("/model-info")
health.raise_for_status()
model_info.raise_for_status()

print("Health:")
print(json.dumps(health.json(), indent=2))
print("
Model info:")
print(json.dumps(model_info.json(), indent=2))

assert model_info.json()["explainability"]["enabled"] is True


## 5. Select an unseen HYAMD AMD validation image

A positive validation example is preferred so the explanation represents an AMD-positive decision when the model agrees.


In [ ]:
import pandas as pd
from PIL import Image

prepared_roots = [
    DATASET_MOUNT / "eyeai_prepared_binary_dataset",
    Path("/kaggle/working/eyeai_prepared_binary_dataset_retfound"),
]
prepared_roots.extend(
    path.parent
    for path in Path("/kaggle/input").rglob("dataset_summary.json")
    if (path.parent / "manifests/hyamd_val.csv").is_file()
)
DATASET_ROOT = next(
    (root for root in prepared_roots if (root / "manifests/hyamd_val.csv").is_file()),
    None,
)
if DATASET_ROOT is None:
    raise FileNotFoundError("The prepared HYAMD dataset was not found.")

validation = pd.read_csv(DATASET_ROOT / "manifests/hyamd_val.csv", dtype={"image_id": str})
positive_validation = validation[validation["binary_label"] == 1]
row = positive_validation.sample(n=1, random_state=42).iloc[0]
image_path = DATASET_ROOT / row["relative_image_path"]
print("Image:", image_path)
print("Truth:", int(row["binary_label"]))


## 6. Compare `/predict` with `/predict-with-explanation`

Both endpoints must return the same TTA probability and decision. The explanation endpoint additionally performs gradient-weighted patch attribution and saves artifacts.


In [ ]:
content_type = {
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png": "image/png",
    ".webp": "image/webp",
    ".tif": "image/tiff",
    ".tiff": "image/tiff",
}[image_path.suffix.lower()]

image_bytes = image_path.read_bytes()
plain = client.post(
    "/predict",
    files={"file": (image_path.name, image_bytes, content_type)},
)
explained = client.post(
    "/predict-with-explanation",
    files={"file": (image_path.name, image_bytes, content_type)},
)
plain.raise_for_status()
explained.raise_for_status()

plain_payload = plain.json()
explained_payload = explained.json()
print(json.dumps(explained_payload, indent=2, ensure_ascii=False))

probability_difference = abs(
    plain_payload["probability"] - explained_payload["probability"]
)
print("
Probability difference:", probability_difference)
assert probability_difference < 1e-5
assert plain_payload["decision"] == explained_payload["decision"]


## 7. Retrieve and display the explanation artifacts

The API returns URLs rather than embedding large images in JSON. The heatmap is masked outside the processed fundus field, and the overlay is rendered on the processed full-fundus image.


In [ ]:
import matplotlib.pyplot as plt

artifact_images = {}
for name in ["original", "processed", "heatmap", "overlay"]:
    artifact = explained_payload["explanation"]["artifacts"][name]
    response = client.get(artifact["url"])
    response.raise_for_status()
    artifact_images[name] = Image.open(io.BytesIO(response.content)).convert("RGB")

figure, axes = plt.subplots(1, 4, figsize=(24, 6))
for axis, name in zip(axes, ["original", "processed", "heatmap", "overlay"]):
    axis.imshow(artifact_images[name])
    axis.set_title(name.title())
    axis.axis("off")

figure.suptitle(
    f"Truth={int(row['binary_label'])} | "
    f"Prediction={explained_payload['label']} | "
    f"P(AMD)={explained_payload['probability']:.4f}"
)
plt.tight_layout()
plt.show()

print("Explanation metrics:")
print(json.dumps(explained_payload["explanation"]["metrics"], indent=2))
print("Explanation warnings:", explained_payload["explanation"]["warnings"])
print("Saved artifact directory:", EXPLANATION_OUTPUT_DIR / explained_payload["request_id"])


## 8. Verify invalid uploads remain rejected

Explainability must preserve the V1 upload-safety contract.


In [ ]:
invalid = client.post(
    "/predict-with-explanation",
    files={"file": ("not-an-image.txt", b"not an image", "text/plain")},
)
print("Invalid upload status:", invalid.status_code)
print(invalid.json())
assert invalid.status_code == 415


## 9. Close the in-process API client

Save the Kaggle output only when you want to preserve the generated smoke-test artifacts. The source image is re-encoded as PNG, so embedded image metadata is not copied.


In [ ]:
client_context.__exit__(None, None, None)
print("FastAPI V2 explainability smoke test completed.")
